# Cboe 免费期权样本：来源字段预检

只取两个官方公开 ZIP 样本并验原始字节、字段/时间格式与 NAS 回读。样本不证明 SOXX 历史覆盖、25 Delta 插值或策略收益。

In [1]:
import io,os,json,hashlib,subprocess,datetime,zipfile,tarfile,tempfile,shlex
from pathlib import Path
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/soxx_option_vendor_sample_preflight_20261023';raw=root/'data/snapshots/soxx_option_vendor_sample_preflight_20261023'
plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and len(plan['exact_GETs'])==2 and not (card/'sample_result.json').exists()
state={'status':'started','GET_attempts':0,'model_fits':0,'native_accounts':0,'SOXX_market_outcome_reads':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];profiles=[]
for spec,stem in zip(plan['exact_GETs'],['option_quotes','option_eod']):
    headers=raw/(stem+'.headers');body=raw/(stem+'.zip')
    start=datetime.datetime.now(datetime.timezone.utc).isoformat()
    cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','45','--max-filesize','10000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A','SOXL Qlib research source audit research@atrl.me','-w','%{http_code} %{url_effective}',spec['url']]
    run=subprocess.run(cmd,text=True,capture_output=True,timeout=55)
    end=datetime.datetime.now(datetime.timezone.utc).isoformat()
    state['GET_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
    code,url=(run.stdout.split(' ',1)+[''])[:2] if run.stdout else ('','')
    receipt={'product':spec['product'],'requested_url':spec['url'],'requested_at_utc':start,'received_at_utc':end,'curl_exit_code':run.returncode,'http_status':code,'final_url':url,'raw_zip_sha256':sha(body) if body.exists() else None,'raw_zip_bytes':body.stat().st_size if body.exists() else None,'headers_sha256':sha(headers) if headers.exists() else None,'stderr':run.stderr[:500]}
    receipts.append(receipt)
    if run.returncode!=0 or code!='200' or not body.exists() or body.stat().st_size>10000000 or not zipfile.is_zipfile(body):break
    with zipfile.ZipFile(body) as z:
        members=[x for x in z.infolist() if not x.is_dir()]
        examples=[]
        for member in members[:20]:
            entry={'name':member.filename,'bytes':member.file_size}
            if member.file_size<=50000000 and member.filename.lower().endswith(('.csv','.txt')):
                with z.open(member) as src:
                    lines=io.TextIOWrapper(src,encoding='utf-8-sig',errors='replace')
                    entry['first_lines']=[line.rstrip('\r\n')[:1000] for _,line in zip(range(3),lines)]
            examples.append(entry)
        profiles.append({'product':spec['product'],'member_count':len(members),'sample_entries':examples,'all_entry_names':[x.filename for x in members[:100]],'ZIP_uncompressed_bytes':sum(x.file_size for x in members)})
(card/'sample_receipts.json').write_text(json.dumps(receipts,indent=2,ensure_ascii=False)+'\n')
(card/'sample_profiles.json').write_text(json.dumps(profiles,indent=2,ensure_ascii=False)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':x.name,'sha256':sha(x),'bytes':x.stat().st_size} for x in sorted(raw.iterdir()) if x.is_file()],'receipts_sha256':sha(card/'sample_receipts.json'),'profiles_sha256':sha(card/'sample_profiles.json')}
(card/'sample_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-option-vendor-source-') as td:
    archive=Path(td)/'samples.tar.gz'
    with tarfile.open(archive,'w:gz') as t:
        for f in sorted(raw.iterdir()):
            if f.is_file():t.add(f,arcname=f.name)
        t.add(card/'sample_manifest.json',arcname='sample_manifest.json')
    digest=sha(archive)
    artifact=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
    assert artifact['sha256']==digest and artifact['size_bytes']==archive.stat().st_size
    restored=Path(td)/'restored.tar.gz'
    with restored.open('wb') as out:
        run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(artifact['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=120)
    assert run.returncode==0 and sha(restored)==digest
    with tarfile.open(restored,'r:gz') as t:
        names=[x.name for x in t.getmembers() if x.isfile()]
    assert set(names)==set(x['name'] for x in manifest['files'])|{'sample_manifest.json'}
status='two_public_Cboe_ZIP_samples_received_schema_only' if len(receipts)==2 and all(x['curl_exit_code']==0 and x['http_status']=='200' for x in receipts) else 'bounded_sample_partial_or_failed'
result={'status':status,'plan_sha256':manifest['plan_sha256'],'GET_attempts':state['GET_attempts'],'receipts':receipts,'profiles':profiles,'raw_NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'official_historical_start':'2012-01','SOXX_specific_historical_coverage_proven':False,'2011_Stage1_training_coverage_proven':False,'25_Delta_constant_tenor_feature_qualified':False,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0,'purchases':0}
(card/'sample_result.json').write_text(json.dumps(result,indent=2,ensure_ascii=False)+'\n')
state.update(status='completed' if status.startswith('two_public') else 'partial_or_failed',NAS_archive_sha256=digest);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({'status':status,'GET_attempts':state['GET_attempts'],'products':[(x['product'],x['http_status'],x['raw_zip_bytes']) for x in receipts],'members':[(x['product'],x['member_count']) for x in profiles],'NAS':digest})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


KeyError: 'nas_path'

## 中文结论

两份官方免费 ZIP 已在冻结 2 次 GET 内取得，原始字节和表头证据已保存。首次 NAS 上传虽获匹配回执，但笔记本误读不存在的 `nas_path` 字段，未完成回读；后续机械修复见 `nas_path_repair.ipynb`。此首次执行不构成来源资格，0 SOXX 收益读取、拟合、账户或购买。